# Description

In this notebook, I will explore the benchmark Human Eval and using QWen to generate it.

In [1]:
import os 
import sys
import numpy as np 
import pandas as pd 
import time
import re
import io
import re
import ast
import types
import unittest
import importlib
from typing import List, Tuple, Dict, Any, Set
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
model_name = "Qwen/Qwen2.5-7B-Instruct-1M"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

MAX_TOKENS = 1024

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

# 1. Load data

In [3]:
PATH_CSV_DATA = "data/raw_data/human_eval.csv"

In [4]:
df = pd.read_csv(PATH_CSV_DATA)
print(f"Dataframe shape: {df.shape}")
df.sample(1)

Dataframe shape: (164, 5)


,task_id,prompt,canonical_solution,test,entry_point
98,HumanEval/98,"\ndef count_upper(s):\n """"""\n Given a st...","count = 0\n for i in range(0,len(s),2):...",def check(candidate):\n\n # Check some simp...,count_upper


In [28]:
idx = np.random.randint(0, df.shape[0])

code_description = df.loc[idx, "prompt"]
test_case = df.loc[idx, "test"]
entry_point = df.loc[idx, "entry_point"]

print("Code description:")
print(code_description)
print("=" * 20)
print("Test Case:")
print(test_case)

Code description:


def encode_shift(s: str):
    """
    returns encoded string by shifting every character by 5 in the alphabet.
    """
    return "".join([chr(((ord(ch) + 5 - ord("a")) % 26) + ord("a")) for ch in s])


def decode_shift(s: str):
    """
    takes as input string encoded with encode_shift function. Returns decoded string.
    """

Test Case:


METADATA = {}


def check(candidate):
    from random import randint, choice
    import copy
    import string

    letters = string.ascii_lowercase
    for _ in range(100):
        str = ''.join(choice(letters) for i in range(randint(10, 20)))
        encoded_str = encode_shift(str)
        assert candidate(copy.deepcopy(encoded_str)) == str




# 2. Using Qwen to generate sample

## 2.1. Generate code

In [29]:
def extract_function(llm_text):
    code = "Empty code"
    # 1) Grab text between <code>...</code>
    m = re.search(r"<code>\s*(.*?)\s*</code>", llm_text, flags=re.S|re.M)
    if m:
        code = m.group(1)
    else:
    # 2) Optionally, if the model sometimes adds backticks, strip them
        code = re.sub(r"^```(?:python)?\s*|\s*```$", "", code.strip())

    return code

def extract_reasoning(llm_text):
    # 1) Grab text between <reasoning>...</reasoning>
    m = re.search(r"<reasoning>\s*(.*?)\s*</reasoning>", llm_text, flags=re.S|re.M)
    if not m:
        return "No <reasoning> block found"
    reasoning = m.group(1).strip()
    return reasoning

In [30]:
constraints = """
Output only a complete and valid Python code for this function. Do not change the provided function signature.
Do NOT print any markdown or include the test cases in your output.
Wrap your output strictly between the markers:
<code>
... your code ...
</code>
"""

COT = """
Before giving the final code, internally think step-by-step about how to implement the function
(do NOT print this reasoning).
"""


input_prompt = f"""write a complete python function
based on the following description:\n{code_description}.\n
{COT}.
with the following constraints:\n{constraints}
"""

print("Input prompt to:")
print(input_prompt)

Input prompt to:
write a complete python function
based on the following description:


def encode_shift(s: str):
    """
    returns encoded string by shifting every character by 5 in the alphabet.
    """
    return "".join([chr(((ord(ch) + 5 - ord("a")) % 26) + ord("a")) for ch in s])


def decode_shift(s: str):
    """
    takes as input string encoded with encode_shift function. Returns decoded string.
    """
.


Before giving the final code, internally think step-by-step about how to implement the function
(do NOT print this reasoning).
.
with the following constraints:

Output only a complete and valid Python code for this function. Do not change the provided function signature.
Do NOT print any markdown or include the test cases in your output.
Wrap your output strictly between the markers:
<code>
... your code ...
</code>




In [31]:
def generate_response(prompt, max_tokens=MAX_TOKENS):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_tokens
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

In [32]:
output = generate_response(input_prompt)
print("Response:\n", output)

num_out_tokens = len(tokenizer(output).input_ids)
print(f"Number of output tokens: {num_out_tokens}")

Response:
 <code>
def decode_shift(s: str):
    """
    takes as input string encoded with encode_shift function. Returns decoded string.
    """
    return "".join([chr(((ord(ch) - 5 - ord("a")) % 26) + ord("a")) for ch in s])
</code>
Number of output tokens: 65


We can extract the complete code

In [33]:
completed_code = extract_function(output)
print(f"The complete code:\n")
print(completed_code)

The complete code:

def decode_shift(s: str):
    """
    takes as input string encoded with encode_shift function. Returns decoded string.
    """
    return "".join([chr(((ord(ch) - 5 - ord("a")) % 26) + ord("a")) for ch in s])


## 2.2. Evaluate the generated code

In [34]:
import builtins
import typing

def create_namespace():
    ns = {}

    # 1. Standard builtins (print, len, etc.)
    ns.update({k: getattr(builtins, k) for k in dir(builtins)})

    # 2. Install common typing names (List, Optional, etc.)
    for name in typing.__all__:
        ns[name] = getattr(typing, name)

    # 3. (Optional) Add math, random, itertools, etc.
    import math, random, itertools, statistics
    ns.update({
        'math': math,
        'random': random,
        'itertools': itertools,
        'statistics': statistics,
    })

    return ns

In [35]:
def evaluate_asserts(generated_code: str, test_code: str, entry_point: str):
    # ns = {}
    ns = create_namespace()
    
    # 1. Exec both code strings
    exec(generated_code, ns)
    exec(test_code, ns)

    candidate = ns[entry_point]     # the model's function
    check_fn = ns["check"]          # original check() function
    
    # 2. Parse the test code AST
    tree = ast.parse(test_code)

    # 3. Find the check() function body
    check_body = None
    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name == "check":
            check_body = node.body
            break

    if check_body is None:
        raise ValueError("check() function not found.")
    
    # 4. Evaluate each assert individually
    results = []
    for idx, stmt in enumerate(check_body):
        if isinstance(stmt, ast.Assert):
            # Convert AST back to executable code
            code = compile(ast.Module([stmt], type_ignores=[]), "<assert>", "exec")
            try:
                exec(code, {**ns, "candidate": candidate})
                results.append(("pass", None))
            except Exception as e:
                results.append(("fail", repr(e)))

    # 5. Compute pass percentage
    total = len(results)
    passed = sum(1 for r, _ in results if r == "pass")
    percentage = passed / total if total > 0 else 0.0

    return {
        "total_asserts": total,
        "passed": passed,
        "percentage": percentage,
        "detail": results
    }

In [36]:
result = evaluate_asserts(completed_code, test_case, entry_point)
print(result)

{'total_asserts': 0, 'passed': 0, 'percentage': 0.0, 'detail': []}


# 3. Run through all sample

In [ ]:
list_df = []
list_num_out_token = []

start_time = time.time()
for idx in range(df.shape[0]):
    if idx % 10 == 0:
        print(f"Processing idx={idx} / {df.shape[0]}")
    
    try:
        # 1. Prepare input prompt
        code_description = df.loc[idx, "prompt"]
        test_case = df.loc[idx, "test"]
        entry_point = df.loc[idx, "entry_point"]
        
        input_prompt = f"""write a complete python function
        based on the following description:\n{code_description}.\n
        {COT}.
        with the following constraints:\n{constraints}
        """

        output = generate_response(input_prompt)
        completed_code = extract_function(output)
        # reasoning_text = extract_reasoning(output)
        list_num_out_token.append(len(tokenizer(output).input_ids))
        
        # 3. Evaluate the generated code
        result = evaluate_asserts(completed_code, test_case, entry_point)
        total_asserts = result["total_asserts"]
        passed_asserts = result["passed"]
        percentage = result["percentage"]
        
        list_df.append({
            "description": code_description,
            "generated_code": completed_code,
            "test_case": test_case,
            "entry_point": entry_point,
            "total_asserts": total_asserts,
            "passed_asserts": passed_asserts,
            "percentage": percentage,
            # "reasoning_text": reasoning_text,
        })
    except Exception as e:
        print(f"Error at idx={idx}: {e}")
        continue
    
end_time = time.time()
avg_time_per_example = (end_time - start_time) / len(list_df)
print(f"Average time per example: {avg_time_per_example:.2f} seconds")

avg_output_tokens = sum(list_num_out_token) / len(list_num_out_token)
print(f"Average number of output tokens: {avg_output_tokens:.2f}")

Processing idx=0 / 164
Processing idx=10 / 164
Processing idx=20 / 164
Processing idx=30 / 164
Processing idx=40 / 164
Processing idx=50 / 164
Processing idx=60 / 164
Processing idx=70 / 164
Processing idx=80 / 164
Processing idx=90 / 164
Processing idx=100 / 164
Processing idx=110 / 164
Processing idx=120 / 164
Processing idx=130 / 164
Processing idx=140 / 164
Processing idx=150 / 164
Processing idx=160 / 164
Average time per example: 2.68 seconds
Average number of output tokens: 87.61


In [38]:
output_df = pd.DataFrame(list_df)
print(f'Output dataframe shape: {output_df.shape}')
output_df.sample()

Output dataframe shape: (164, 8)


,description,generated_code,test_case,entry_point,total_asserts,passed_asserts,percentage,reasoning_text
105,"\ndef by_length(arr):\n """"""\n Given an a...",def by_length(arr):\n # Filter numbers betw...,def check(candidate):\n\n # Check some simp...,by_length,7,3,0.428571,No <reasoning> block found


In [39]:
# Save to CSV
output_df.to_csv("data/generated/human_eval_generated_qwen_COT.csv", index=False)

## 3.1. Check generated code

In [40]:
average_percentage = output_df["percentage"].mean()
print(f"Average pass percentage over all samples: {average_percentage:.2%}")    

Average pass percentage over all samples: 92.05%
